In [ ]:
import os, sys, shutil, subprocess, time
from pathlib import Path
import torch

BASE = Path('/kaggle/working')
assert BASE.is_dir(), '请在 Kaggle Notebook 中运行'
assert torch.cuda.is_available(), '当前没有可用 GPU；请在 Settings 里开启 Accelerator → GPU，然后重跑本格'
print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print('磁盘剩余 GiB:', round(shutil.disk_usage(BASE).free / 2**30, 2))

In [ ]:
import subprocess, shutil, os, sys
from pathlib import Path

MORTAL_COMMIT = '0cff2b52982be5b1163aa9a62fb01f03ce91e0d2'
repo = BASE / 'Mortal'
if not (repo / '.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/Equim-chan/Mortal.git', str(repo)], check=True)
subprocess.run(['git', '-C', str(repo), 'fetch', 'origin', MORTAL_COMMIT], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', '--detach', MORTAL_COMMIT], check=True)
actual_commit = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip()
assert actual_commit == MORTAL_COMMIT, actual_commit

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'toml', 'tqdm', 'tensorboard', 'huggingface_hub'], check=True)
rustc = shutil.which('rustc')
rust_version = subprocess.check_output([rustc, '--version'], text=True).split()[1] if rustc else '0.0.0'
major_minor = tuple(map(int, rust_version.split('.')[:2]))
rustup = Path.home() / '.cargo' / 'bin' / 'rustup'
if major_minor < (1, 85) or not shutil.which('cargo'):
    if not rustup.exists():
        subprocess.run(['bash', '-lc', 'curl --proto "=https" --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal'], check=True)
    subprocess.run([str(rustup), 'update', 'stable'], check=True)
    cargo_cmd = [str(Path.home() / '.cargo' / 'bin' / 'cargo'), '+stable']
else:
    cargo_cmd = [shutil.which('cargo')]
print(subprocess.check_output(cargo_cmd + ['--version'], text=True).strip())
subprocess.run(cargo_cmd + ['build', '--locked', '-p', 'libriichi', '--lib', '--release'], cwd=repo, check=True)
built = repo / 'target' / 'release' / 'libriichi.so'
assert built.is_file(), f'编译产物不存在: {built}'
shutil.copy2(built, repo / 'mortal' / 'libriichi.so')
sys.path.insert(0, str(repo / 'mortal'))
from libriichi.arena import OneVsThree
print('libriichi 导入成功；Mortal commit:', actual_commit)

In [ ]:
import hashlib
from huggingface_hub import hf_hub_download

CHECKPOINTS = {
    '582500': ('Yuchen1457/mortal-582500', 'mortal_582500.pth', '738e0d6e3c0ce9671629554ad39abd147d2ffbac676e80b194c83f2acc0fea20'),
    '298k': ('VoidShine/mortal-298k', 'mortal_298k.pth', 'bfb3a6c072aa0bfd4171a9cdc77cb6c02ae42cde920843f9e5784394f23447d8'),
}
model_dir = BASE / 'checkpoints'
model_dir.mkdir(exist_ok=True)
model_paths = {}
for label, (repo_id, filename, expected_sha) in CHECKPOINTS.items():
    path = Path(hf_hub_download(repo_id=repo_id, filename=filename, local_dir=model_dir / label))
    with path.open('rb') as f:
        digest = hashlib.file_digest(f, 'sha256').hexdigest()
    assert digest == expected_sha, f'{label} SHA-256 不符: {digest}'
    model_paths[label] = path
    print(label, 'OK', path.stat().st_size, 'bytes', digest)

In [ ]:
import hashlib
import numpy as np
from model import Brain, DQN
from engine import MortalEngine
from libriichi.arena import OneVsThree

def load_engine(label):
    state = torch.load(model_paths[label], weights_only=True, map_location='cpu')
    cfg = state['config']
    version = cfg['control'].get('version', 1)
    brain = Brain(version=version, conv_channels=cfg['resnet']['conv_channels'],
                  num_blocks=cfg['resnet']['num_blocks']).eval()
    dqn = DQN(version=version).eval()
    brain.load_state_dict(state['mortal'])
    dqn.load_state_dict(state['current_dqn'])
    print(label, 'version', version, 'channels', cfg['resnet']['conv_channels'],
          'blocks', cfg['resnet']['num_blocks'])
    return MortalEngine(brain, dqn, is_oracle=False, version=version,
                        device=torch.device('cuda:0'), enable_amp=True,
                        enable_rule_based_agari_guard=True, name=label)

engines = {label: load_engine(label) for label in CHECKPOINTS}
KEY = int.from_bytes(hashlib.sha256(b'Mortal 582500 vs 298k 2026-09-25').digest()[:8], 'big')
arena = OneVsThree(disable_progress_bar=False, log_dir=None)
pilot = {}
for challenger, champion in [('582500', '298k'), ('298k', '582500')]:
    t0 = time.monotonic()
    ranks = list(arena.py_vs_py(challenger=engines[challenger], champion=engines[champion],
                                seed_start=(9900, KEY), seed_count=10))
    elapsed = time.monotonic() - t0
    assert sum(ranks) == 40, ranks
    pilot[challenger] = {'ranks': ranks, 'seconds': elapsed}
    print(challenger, '挑战者试跑:', ranks, '耗时', round(elapsed, 1), '秒')
print('按试跑速度，正式测试每方向 8000 场预计总小时数:',
      round(200 * sum(x['seconds'] for x in pilot.values()) / 3600, 2))

In [ ]:
import json, time, os

import base64, zlib
RESULT_PATH = BASE / 'mortal_benchmark_results.jsonl'
if not RESULT_PATH.exists():
    snapshot = zlib.decompress(base64.b64decode('eNrNXNFqXEcMfe9XGD+bYaSRZqT+SinFdtZtSGOXJH0oof/emd27dnNnXAinWBtYCDf23hw0ko6OpPl6ff/b7e+/Hx5/PXy6/vHqWo015+ubq/H84x/vnx7HU3b7MJ59Phze/fL5y+2nL/0p5f7n/PD+6c/H08P+5MPhr/5XF24kUogLCXGW1sbXPn38+H785HW+f3jgO+3fzXcHvSOq5fbWbys/3GV6yOX+4HTI73i8+NPt44fP/Zd+Gt9vN1f15orqz8eX3z89vhv/1Gpq9e8fvu4Anf/r/4bzAvICAJWbK+1oaHx2iDgJlRnS99mIQmw0Pr6AZCk3R60UAElOcPpr2g5R6cdOUSPx2yOiE5yBTCdIZAU1UgwkO9mJp9igCntSeXNEbYt0C0fS1AR2pLdHRHw0kC0QlVS4oTaSkGi3hYZFANemqJFiIJ3jg+8geWoF9iR9c0j1ZKT+TbYD1FIVmDeEAHoO4eVbSNVSa4zaqIYcu3bKSPtTJ0mMUSPVkGjXNlBTRnKBaUMLoavjU2dHyqkY7EgtjAgdY/i3kKwmyY4ayd4ckj370YJ/t2qokSzKkcbJ2x0740R4sPMYR8rrkoKSOxzt3h6Sb9x7dqTBv01AI1GOQLQR1ik0dLaKMrsgQGfWkCf+bQLbiEIglVN0oJkHOVojBSCyM62beVBOjQm1EYcdO55Zw0ixKGugIPnkpDdMjiQpK1rIUgkjQm2hn2gy3Eoh+skWHiYi1BI3Q40kIbShrjUuTiXD0U7Cqr5FIds9idEaiTSErG4VxVzHloIyuwhE9KJxzdGuGRztakRKsi005KlGYhXUSDXs2NmSf3tBC1lqYZDqItx1Ag4bqYVwOz7ZaAfIerAj2EYWVsj6qjT32lAbWYi0Wl5R9D2h4jd5jJ6/1Uh5Yg1WYNbgYYIQzznWUl4J+t9lJM4XVVHY/yCtBkCyF0ea20iZUU/igDkAPbcvbTYR6kZMUeloSew8VVQOYo4o+Y4q8YIxeFJCTcQh+XWbeCoTBcoZVYM4ZgagvjLyJJ2oopQhAFJ9id3zME2G+xMsUdL3InbbGD1BtW+WSxpLs5LEUe7NGqad6JSPPCeCxyE5RmmoazWolxMKNyi4hp27VUrKqb8ItVKIeLJNRO5Lvg4I5t4tanLQFjnWk6D98ghAW2CgxZkbTBVmDRbWQypzv1yTFDjWWYiRzjyoTH7khKqq7DFFLC8jQ0+xBFdIHkW/bdlC0oqShpLD6LfMRawmgnWGAES6cdWFmN8dqb8QNVJMYU6vapACaw0BkDbJbpVkLXlFm7ElaAigH77yyqIL7EocYiRdE4cewQmPdyVMWF03kQguZQMgPe8bLAKepKJwwJOoDSub5Yaxu4OaSIJmcH3eU+wZSeGh4qJBszS2WLzUpARbSENO3KZ/18mHqsP5KKQs33piq6Ivo0JDCZvSkFWXr8HzW6WFHDp5dUhaFNW+AyD5S8dlHiluglaxxcKq2LbsirUGO1IMpLYs+oySCEwYPKy5XKbwfZyygx3Jo6bs2qJA4lQcdSTJF0TqJGVYDZKYiQZdATqWsKgXCV1S49KTwqpqAKL/2OcbKy6OtieEw4TivNgjLZ00oOQ7AtJGgnT2JEqksJFCdg2eR2l2YpBbsoYmJAm7rOFIhXaIJDnDRopZn9jEk7k/gc/YBSHaOn0+jQAwrNeJXlB/wktyeOhbNGz+9qhCTo7U4I651IvSiTk1h610YRsh3mAjBV3WcB6omYa4jOFwFyY2HMvZnSe1RIyOB4ld0G5sL5LM0Xu4JOayhi0t2cxWG7rRJx42wrVaYLZUHXYkD2swzwsUY84O3ujTsCkAn4ukkWRhGwUgonoWhPZEaNwMiVoo7GIDX8w7ac+wKGcIQHTORbaqJyq8P6F8QezbKLmiy8tBiNZzJ1bxFRctEW7kr4xI16TwZXZaLmnZoCZGtydUwq5wWbT5KNeU4SpWYzZCdKv69pA4FUILJNVLuruzp1gzdPI7ANL5ctW5Lqecez0Bx7sa4Uy2vsVlQGJGl/oCII30WpcrIcOXWoVDXgvRv/Nru0htDKz+Ay7iOWU='))
    assert hashlib.sha256(snapshot).hexdigest() == '643d23330fbd2bc5cac138f7613d50557bbcf8ed81701bffc904d73d4848a8fb'
    RESULT_PATH.write_bytes(snapshot)
    print('Restored checkpoint rows:', len(snapshot.splitlines()))
START_SEED = 10000
SEEDS_PER_BATCH = 10
TARGET_SEEDS = 2000  # 8000 games per direction
MAX_WALL_SECONDS = 11 * 3600
assert TARGET_SEEDS % SEEDS_PER_BATCH == 0

def saved_rows():
    if not RESULT_PATH.exists():
        return []
    with RESULT_PATH.open(encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

def append_row(row):
    with RESULT_PATH.open('a', encoding='utf-8') as f:
        f.write(json.dumps(row, ensure_ascii=False) + chr(10))
        f.flush()
        os.fsync(f.fileno())

existing = saved_rows()
for row in existing:
    assert row['commit'] == MORTAL_COMMIT and row['key'] == KEY and row['seed_count'] == SEEDS_PER_BATCH, 'Incompatible saved result'
    assert START_SEED <= row['seed_start'] < START_SEED + TARGET_SEEDS, 'Incompatible seed range'
    assert row['challenger'] in CHECKPOINTS
done = {(r['seed_start'], r['challenger']) for r in existing}
assert len(done) == len(existing), 'Duplicate result rows'
started = time.monotonic()
for seed_start in range(START_SEED, START_SEED + TARGET_SEEDS, SEEDS_PER_BATCH):
    if time.monotonic() - started >= MAX_WALL_SECONDS:
        print('Reached 11-hour limit; results saved')
        break
    for challenger, champion in [('582500', '298k'), ('298k', '582500')]:
        if (seed_start, challenger) in done:
            continue
        t0 = time.monotonic()
        ranks = list(arena.py_vs_py(challenger=engines[challenger], champion=engines[champion],
                                    seed_start=(seed_start, KEY), seed_count=SEEDS_PER_BATCH))
        assert sum(ranks) == 4 * SEEDS_PER_BATCH, ranks
        row = {'challenger': challenger, 'champion': champion,
               'seed_start': seed_start, 'seed_count': SEEDS_PER_BATCH,
               'key': KEY, 'commit': MORTAL_COMMIT,
               'ranks': ranks, 'seconds': round(time.monotonic() - t0, 3)}
        append_row(row)
        done.add((seed_start, challenger))
        print(f"{challenger} vs {champion}: seed {seed_start}, ranks {ranks}, {row['seconds']:.1f} sec", flush=True)
print('Completed batches:', len(done), '/', 2 * TARGET_SEEDS // SEEDS_PER_BATCH)
print('Results file:', RESULT_PATH)

In [ ]:
import numpy as np
import pandas as pd

rows = saved_rows()
by_key = {(r['seed_start'], r['challenger']): r for r in rows}
paired_seeds = sorted(s for s in range(START_SEED, START_SEED + TARGET_SEEDS, SEEDS_PER_BATCH)
                      if (s, '582500') in by_key and (s, '298k') in by_key)
assert paired_seeds, 'No completed paired batches'
a = np.asarray([by_key[(s, '582500')]['ranks'] for s in paired_seeds], dtype=np.int64)
b = np.asarray([by_key[(s, '298k')]['ranks'] for s in paired_seeds], dtype=np.int64)
ranks_axis = np.array([1, 2, 3, 4])
pt_axis = np.array([90, 45, 0, -135])

def metrics(counts):
    n = counts.sum()
    return {'对局数': int(n), '平均顺位': counts @ ranks_axis / n,
            '一位率': counts[0] / n, '二位率': counts[1] / n,
            '三位率': counts[2] / n, '四位率': counts[3] / n,
            'Rank PT': counts @ pt_axis / n}

raw = pd.DataFrame({k: metrics(v.sum(axis=0)) for k, v in [('582500', a), ('298k', b)]}).T
raw.to_csv(BASE / 'mortal_benchmark_summary.csv', encoding='utf-8-sig')
summary = raw.copy()
summary['对局数'] = summary['对局数'].astype(int)
summary['平均顺位'] = summary['平均顺位'].map(lambda x: f'{x:.4f}')
for col in ['一位率', '二位率', '三位率', '四位率']:
    summary[col] = summary[col].map(lambda x: f'{x:.2%}')
summary['Rank PT'] = summary['Rank PT'].map(lambda x: f'{x:+.3f}')
display(summary)
print('Paired batches:', len(paired_seeds), 'games per model:', 4 * SEEDS_PER_BATCH * len(paired_seeds))
if len(paired_seeds) >= 5:
    rng = np.random.default_rng(20260925)
    indices = rng.integers(0, len(paired_seeds), size=(5000, len(paired_seeds)))
    aa, bb = a[indices].sum(axis=1), b[indices].sum(axis=1)
    pt_diff = (aa @ pt_axis / aa.sum(axis=1)) - (bb @ pt_axis / bb.sum(axis=1))
    rank_diff = (aa @ ranks_axis / aa.sum(axis=1)) - (bb @ ranks_axis / bb.sum(axis=1))
    print('582500 - 298k Rank PT diff, 95% CI:', round(raw.loc['582500', 'Rank PT'] - raw.loc['298k', 'Rank PT'], 3), np.quantile(pt_diff, [0.025, 0.975]).round(3))
    print('582500 - 298k mean rank diff, 95% CI:', round(raw.loc['582500', '平均顺位'] - raw.loc['298k', '平均顺位'], 4), np.quantile(rank_diff, [0.025, 0.975]).round(4))
print('Saved:', RESULT_PATH, BASE / 'mortal_benchmark_summary.csv')

import json

expected_batches = TARGET_SEEDS // SEEDS_PER_BATCH
completed = len(paired_seeds) == expected_batches
status = {
    'completed': completed,
    'paired_batches': len(paired_seeds),
    'target_batches': expected_batches,
    'games_per_model': 4 * SEEDS_PER_BATCH * len(paired_seeds),
    'target_games_per_model': 4 * TARGET_SEEDS,
    'saved_rows': len(rows),
}
(BASE / 'mortal_benchmark_status.json').write_text(
    json.dumps(status, ensure_ascii=False, indent=2), encoding='utf-8')
print('BENCHMARK COMPLETE' if completed else 'BENCHMARK INCOMPLETE; resume from saved JSONL')
print(status)
